# Vectorstore population & retrieval

Ingests chunk embeddings via `FaoRepository` and `TellusDataSource` (they run the data-lake pipelines including `embed_chunks`), then queries the hybrid vectorstore.

**Prerequisites**

1. Local Atlas MongoDB: `docker compose up -d` (see README). Host port **27018**.
2. `.env` with `MONGO_USERNAME`, `MONGO_PASSWORD`, `AWS_BEDROCK_API_KEY`, and `TELLUS_BEARER_TOKEN`.
3. Start Jupyter from the project root (or run the setup cell, which `chdir`s there).

Keep crawl / Tellus limits small for interactive runs.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv

# Resolve project root whether the kernel cwd is repo root or notebooks/
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
os.chdir(ROOT)
load_dotenv(ROOT / ".env")
print("Working directory:", ROOT)

# Keep interactive runs bounded (PdfCrawlConfig / TellusConfig read these).
# Depth is 0-based: seed=0, first-hop pages=1, PDF links from those=2.
os.environ["PDF_MAX_PDFS"] = "10"
os.environ["PDF_MAX_URL_DEPTH"] = "2"
os.environ["PDF_MAX_URLS"] = "100"
os.environ["PDF_MAX_URLS_PER_PAGE"] = "8"
os.environ["TELLUS_MAX_RESULTS"] = "5"


# Register stages (get_stage) and data sources (get_data_source).
import fao_impact_monitor.data_lake.stages
import fao_impact_monitor.data_source  # noqa: F401
from fao_impact_monitor.config import get_config
from fao_impact_monitor.data_lake.mongo import connect_data_lake, get_mongo_config
from fao_impact_monitor.data_lake.vectorstore import (
    ChunkEmbedding,
    VectorStore,
    ensure_indexes,
)
from fao_impact_monitor.data_source import (
    DataSourceConfig,
    FaoRepositoryDataSourceConfig,
    get_data_source,
)
from fao_impact_monitor.metric import Metric

config = get_config()
print("Embedding model:", config.vector_store.embedding_model)
print("PDF max_pdfs:", config.pdf_crawl.max_pdfs)
print("PDF max_url_depth:", config.pdf_crawl.max_url_depth)
print("PDF max_urls:", config.pdf_crawl.max_urls)
print("Tellus max_results:", config.tellus.max_results)

COUNTRY_ISO3 = "KEN"
METRIC = Metric(
    name="El Niño agricultural impacts",
    description="El Niño impacts on agriculture, crops, and food security",
    example="El Niño reduced maize yields in Kenya.",
    unit="qualitative",
    data_sources=[],
)
print("Country:", COUNTRY_ISO3)
print("Metric:", METRIC.name)

## Connect MongoDB and initialize Beanie

In [ ]:
mongo = get_mongo_config()
client = await connect_data_lake(mongo)

from pymongo.errors import PyMongoError

collection = ChunkEmbedding.get_pymongo_collection()
try:
    await ensure_indexes(collection)
    print("Search indexes ensured.")
except PyMongoError as exc:
    # Indexes may already exist from a previous run.
    print(f"ensure_indexes note: {exc}")

print("Connected to", mongo.db_name, f"at {mongo.host}:{mongo.port}")

## Ingest via `FaoRepository`

Calls `FaoRepository.get_data`, which runs `pdf_crawl` and cascades discovered PDFs through `pdf_process` (extract → country detect → embed).

In [ ]:
fao_config = FaoRepositoryDataSourceConfig(
    source="FaoRepository",
    url=(
        "https://www.fao.org/emergencies/resources-repository/publications/publications-result/en?indexCatalogue=search-index-emergencies&wordsMode=AllWords&fallbacklang=en&searchMode=all&contentTypes=FAOResources.FaoResourcesPublications&searchQuery=El%20Ni%C3%B1o"
    ),
)

fao = get_data_source("FaoRepository")
fao_results = await fao.get_data(METRIC, fao_config, COUNTRY_ISO3)

print(f"FaoRepository returned {len(fao_results)} PDF result(s)")
for result in fao_results:
    print(f"  - {result.title!r}")
    print(f"    {result.url}")

## Ingest via `TellusDataSource`

Calls `TellusDataSource.get_data`, which searches Tellus from the metric description and runs `tellus_process` (fetch → country detect → embed) per matched document.

In [ ]:
tellus = get_data_source("Tellus")
tellus_results = await tellus.get_data(
    METRIC,
    DataSourceConfig(source="Tellus"),
    COUNTRY_ISO3,
)

print(f"Tellus returned {len(tellus_results)} document result(s)")
for result in tellus_results:
    print(f"  - {result.title!r}")
    print(f"    {result.url}")

## Ingestion stats

Unique documents and chunks currently stored in the embeddings collection.

In [ ]:
embeddings = await ChunkEmbedding.find_all().to_list()
unique_document_ids = {row.document_id for row in embeddings}

n_chunks = len(embeddings)
n_documents = len(unique_document_ids)

print(f"Unique documents ingested: {n_documents}")
print(f"Chunks ingested: {n_chunks}")

by_source: dict[str, int] = {}
by_type: dict[str, int] = {}
for row in embeddings:
    src = row.document_source or "(none)"
    by_source[src] = by_source.get(src, 0) + 1
    dtype = str(row.document_type)
    by_type[dtype] = by_type.get(dtype, 0) + 1

print("Chunks by document_source:", by_source)
print("Chunks by document_type:", by_type)

## Query the vectorstore

Hybrid BM25 + vector search (`$rankFusion`). Newly created Atlas indexes can take a short while to become queryable — re-run this cell if search returns empty right after the first ingest.

In [ ]:
QUERY = "How did El Niño affect crop yields and food security in Kenya?"
COUNTRIES_FILTER = [COUNTRY_ISO3]  # set to None to disable country filter
LIMIT = 8

store = VectorStore()
hits = await store.search(
    QUERY,
    countries_iso3=COUNTRIES_FILTER,
    limit=LIMIT,
)

print(f"Query: {QUERY!r}")
print(f"Countries filter: {COUNTRIES_FILTER}")
print(f"Hits: {len(hits)}\n")

for i, hit in enumerate(hits, start=1):
    preview = " ".join(hit.chunk_text.split())
    if len(preview) > 280:
        preview = preview[:277] + "…"
    print(f"--- hit {i} score={hit.score!r} ---")
    print(f"title: {hit.document_title!r}")
    print(f"url: {hit.document_url}")
    print(f"type: {hit.document_type}  source: {hit.document_source}")
    print(f"countries: {hit.countries_iso3}")
    print(f"chunk[{hit.chunk_index}]: {preview}\n")